In [ ]:
!pip install transformers gradio -q
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging
import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity_error()

MODEL = 'facebook/blenderbot_small-90M'

print('Loading chatbot model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
print('Chatbot ready!')


Loading chatbot model...


config.json:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/964k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/345k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  350MB            

Loading weights:   0%|          | 0/347 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/311 [00:00<?, ?B/s]

Chatbot ready!


In [ ]:

def chat(user_message, history):
    if not user_message.strip():
        return "Please type a message.", history

    history.append({
        "role": "user",
        "content": user_message
    })

    # Prepare conversation context without sep_token
    conversation = "\n".join(
        message["content"] for message in history[-6:]
    )

    inputs = tokenizer(
        conversation,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    import torch

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            top_k=50,
            temperature=0.7
        )

    bot_reply = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    if not bot_reply:
        bot_reply = "I'm here to help! Could you ask me another question?"

    history.append({
        "role": "assistant",
        "content": bot_reply
    })

    return bot_reply, history


In [ ]:

history = []

reply, history = chat("Hi! How are you?", history)

print("Bot:", reply)


Bot: i ' m good . just got back from a long day at work . how are you ?


In [ ]:

import gradio as gr

# ==========================================
# CHAMPBOT - CUSTOM WEBPAGE DESIGN
# Compatible with Gradio Chatbot message format
# ==========================================

CSS = """
body, .gradio-container {
    background: #eef3ff !important;
    font-family: 'Segoe UI', sans-serif !important;
}

.gradio-container {
    max-width: 1500px !important;
    padding: 12px !important;
}

#main-panel {
    border: 1px solid #d9e4ff;
    border-radius: 20px;
    overflow: hidden;
    background: white;
    box-shadow: 0 8px 35px rgba(44, 73, 160, 0.08);
}

#sidebar {
    background: linear-gradient(160deg, #17234f, #202c69 70%, #3539bd);
    color: white;
    padding: 26px 18px;
    min-height: 720px;
}

#brand {
    font-size: 28px;
    font-weight: 800;
    color: white;
}

#tagline {
    font-size: 15px;
    line-height: 1.7;
    color: #dce6ff;
    margin: 10px 0 28px 0;
}

#header {
    background: #f0f5ff;
    padding: 12px 22px;
    border-bottom: 1px solid #e6edff;
}

#header h1 {
    color: #142452;
    font-size: 27px;
}

#status {
    color: #248b76;
    text-align: right;
    font-size: 14px;
}

#chat-window {
    background: white;
    padding: 12px;
}

#chat-window .message {
    border-radius: 18px !important;
}

#composer {
    background: #f1f5ff;
    border: 1px solid #e1eaff;
    border-radius: 18px;
    padding: 10px;
    margin: 10px;
}

#message-box textarea {
    border: 1px solid #dfe8ff !important;
    border-radius: 15px !important;
    background: white !important;
    padding: 14px !important;
    font-size: 16px !important;
}

#send-btn button {
    background: linear-gradient(135deg, #4775ff, #5142ff) !important;
    color: white !important;
    border: none !important;
    border-radius: 15px !important;
    min-height: 50px;
    font-weight: 600;
}

#clear-btn button,
#new-chat-btn button {
    background: #edf2ff !important;
    color: #263d7a !important;
    border: 1px solid #dbe5ff !important;
    border-radius: 12px !important;
}

#help-card {
    border: 1px solid #4253a0;
    border-radius: 15px;
    padding: 15px;
    margin-top: 30px;
    color: #e4ebff;
    line-height: 1.7;
}

footer {
    display: none !important;
}
"""


# ==========================================
# CHAT LOGIC
# ==========================================

def reply_in_page(message, history):
    history = history or []

    if not message or not message.strip():
        return history, ""

    user_message = message.strip()

    try:
        # Convert Gradio messages to the model's history format.
        model_history = []

        for item in history:
            if not isinstance(item, dict):
                continue

            role = item.get("role")
            content = item.get("content", "")

            if role in ("user", "assistant") and isinstance(content, str):
                model_history.append({
                    "role": role,
                    "content": content
                })

        # Use the already loaded model and existing chat() function.
        # Exclude the initial greeting from model context.
        model_history = [
            item for item in model_history
            if item.get("content") != (
                "Hello! I'm ChampBot! 😊\n"
                "Your AI companion, ready to help!\n"
                "What would you like to know or do today?"
            )
        ]

        reply, _ = chat(user_message, model_history)

        if not isinstance(reply, str) or not reply.strip():
            reply = "I couldn't generate a response. Please try again."

    except Exception as error:
        print("Chatbot error:", repr(error))
        reply = (
            "Sorry, I couldn't process that message. "
            "Please try again."
        )

    updated_history = history + [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": reply}
    ]

    return updated_history, ""


def clear_conversation():
    return [], ""


def start_new_chat():
    return [], ""


# ==========================================
# WEBPAGE
# ==========================================

WELCOME = (
    "Hello! I'm ChampBot! 😊\n"
    "Your AI companion, ready to help!\n"
    "What would you like to know or do today?"
)

with gr.Blocks(
    title="ChampBot - Your AI Companion",
    css=CSS,
    theme=gr.themes.Soft(primary_hue="indigo")
) as app:

    with gr.Row(elem_id="main-panel", equal_height=True):

        # LEFT SIDEBAR
        with gr.Column(
            scale=1,
            min_width=220,
            elem_id="sidebar"
        ):
            gr.HTML("""
                <div style="text-align:center; font-size:58px;">
                    🤖
                </div>
                <div id="brand">ChampBot</div>
                <div id="tagline">
                    Your AI companion,<br>
                    ready to help! ✨
                </div>
            """)

            gr.Markdown("### 💬 Chat")

            new_chat_btn = gr.Button(
                "➕  New Chat",
                elem_id="new-chat-btn"
            )

            gr.Markdown("### ⚙️ Settings")
            gr.Markdown("### ⓘ About")

            gr.HTML("""
                <div id="help-card">
                    <div style="font-size:28px;">💡</div>
                    <b>Ask me anything!</b><br>
                    Ask questions, learn new things,
                    or explore ideas with ChampBot.
                </div>
            """)

        # MAIN CHAT PANEL
        with gr.Column(
            scale=3,
            min_width=320
        ):
            with gr.Row(elem_id="header"):
                gr.Markdown("# ChampBot")
                gr.Markdown("🟢 Online", elem_id="status")

            chatbot = gr.Chatbot(
                value=[
                    {
                        "role": "assistant",
                        "content": WELCOME
                    }
                ],
                height=520,
                elem_id="chat-window",
                label="Conversation"
            )

            with gr.Row(elem_id="composer"):
                message = gr.Textbox(
                    placeholder="Type your message...",
                    show_label=False,
                    lines=1,
                    max_lines=4,
                    scale=8,
                    elem_id="message-box"
                )

                send_btn = gr.Button(
                    "➤ Send",
                    variant="primary",
                    scale=1,
                    elem_id="send-btn"
                )

            clear_btn = gr.Button(
                "🗑️ Clear Chat",
                elem_id="clear-btn"
            )

    # Send button
    send_btn.click(
        fn=reply_in_page,
        inputs=[message, chatbot],
        outputs=[chatbot, message]
    )

    # Enter key
    message.submit(
        fn=reply_in_page,
        inputs=[message, chatbot],
        outputs=[chatbot, message]
    )

    # Clear conversation
    clear_btn.click(
        fn=clear_conversation,
        inputs=[],
        outputs=[chatbot, message]
    )

    # Start a new conversation
    new_chat_btn.click(
        fn=start_new_chat,
        inputs=[],
        outputs=[chatbot, message]
    )


# ==========================================
# LAUNCH
# ==========================================

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://73797ebd56c7acfe1a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
